# Problem 007

https://projecteuler.net/problem=7


## Brute force approach

A direct approach is to test each positive integer in order and count how many primes have appeared so far. For each candidate $n$, we check whether it is divisible by any integer between $2$ and $\sqrt n$, since if

$$
n = ab,
$$

then at least one of $a$ or $b$ must be at most $\sqrt n$. So if $n$ has no divisors up to $\sqrt n$, then it has no non-trivial divisors at all.

This gives a simple first solution.

Unfortunately, this leads to a runtime error on my laptop. So we are motivated to improve.

In [1]:
def PrimeTrial(n: int) -> bool: 
    if n < 2: 
        return False 
    for d in range(2, int(n**0.5) + 1): 
        if n % d == 0: 
            return False 
        return True

In [2]:
def nthPrimeBruteForce(k: int) -> int: 
    count = 0 
    n = 1 
    while count < k: 
        n += 1 
        if is_prime_trial(n): 
            count += 1 
    return nthPrimeBruteForce(10001)

nthPrimeBruteForce(10001)

NameError: name 'is_prime_trial' is not defined


## Complexity
Let $N$ be the size of the prime we eventually find. For this problem, $N$ is the $10001$ st prime.

The brute force method tests each candidate up to $N$. For each candidate $n$, trial division checks possible divisors up to $\sqrt n$. So the rough time complexity is

$$
O(N\sqrt N).
$$

The space complexity is

$$
O(1),
$$

because we only store the current candidate and the number of primes found so far.

This is simple and works for this problem, but it repeats a lot of unnecessary divisibility checks.

### Improved brute force using known primes

A small improvement is to store the primes found so far. Then, when testing a new candidate $n$, we only divide by known primes up to $\sqrt n$, instead of checking every integer.

This is still trial division, but it is noticeably better because composite divisors do not need to be checked.


In [ ]:
def PrimeTrialUsingKnown(n: int, primes: list[int]) -> bool: 
    for p in primes: 
        if p * p > n: 
            break 
        if n % p == 0: 
            return False 
        return True

In [ ]:
def nthPrimeUsingKnown(k: int) -> int: 
    primes = [] 
    n = 2 
    while len(primes) < k: 
        if PrimeTrialUsingKnown(n, primes): 
            primes.append(n)
            n += 1
    return primes[-1] 
nthPrimeUsingKnown(10001)

KeyboardInterrupt: 

## Mathematical observation
The deeper idea is that primality testing and sieving are different perspectives.

Trial division asks:

$$
\text{Is this number divisible by any smaller prime?}
$$

A sieve instead works by generating composites. Once a prime $p$ is known, its multiples

$$
2p, 3p, 4p, \dots
$$

are known to be composite.

In the Sieve of Eratosthenes, we mark off multiples of each prime. For a fixed bound $N$, this gives time complexity roughly

$$
O(N \log \log N)
$$

and space complexity

$$
O(N).
$$

For this notebook, I use an incremental sieve. Instead of choosing an upper bound first, the algorithm processes numbers in increasing order. It keeps a dictionary of composites that are known to be coming up.

When a number is reached:

if it is not in the dictionary, it has not been generated as a composite, so it is prime;
if it is in the dictionary, it is composite, and the primes responsible for generating it are advanced to their next multiples.

This produces primes and composites in order.



In [ ]:
#Incremental sieve approach
def nthPrimeIncrementalSieve(k: int) -> int:
    primes = []
    composites = {}

    n = 2

    while len(primes) < k:
        if n not in composites:
            primes.append(n)

            # The first composite we need to mark using n is n^2.
            # Smaller multiples of n have already been generated by smaller primes.
            composites[n * n] = [n]

        else:
            for p in composites[n]:
                next_composite = n + p

                while next_composite in composites:
                    next_composite += p

                composites.setdefault(next_composite, []).append(p)

            del composites[n]

        n += 1

    return primes[-1]


nthPrimeIncrementalSieve(10001)



104743

### Complexity of the sieve approach

Let $N$ be the value of the $k$ th prime. The incremental sieve processes numbers in order up to $N$.

For generating all primes up to $N$, sieve-based methods have approximately

$$
O(N \log \log N)
$$

time complexity, depending on implementation details.

The dictionary-based incremental sieve avoids storing a full boolean array of length $N$. However, it does store active composite information and the list of primes found so far.

If we store all primes found, the space complexity is approximately

$$
O(\pi(N)),
$$

where $\pi(N)$ is the number of primes less than or equal to $N$.



## Lessons learned
- Trial division is simple and useful as a first approach.

- Checking divisors only up to $\sqrt n$ is enough.Storing known primes avoids unnecessary composite divisor checks.

- A sieve works by generating composites rather than repeatedly testing candidates from scratch.

- The incremental sieve is a natural way to generate primes in order without choosing an upper bound in advance.

- For Project Euler problems, a good solution is not just about getting the answer; it is about recognising the mathematical structure behind the computation.
